# Credit Card Fraud Detection — Exploratory Data Analysis (EDA)

## 📌 Key Findings & Executive Summary
1. **Severe Class Imbalance**: The dataset contains **284,807 transactions with only 492 frauds (~0.172% positive class)**. A naive model predicting all transactions as legitimate achieves **99.83% accuracy**, making accuracy completely useless as an optimization or reporting metric. Precision-Recall AUC (AUC-PR) and the Kolmogorov-Smirnov (KS) statistic must be prioritized.
2. **Heavy-Tailed Transaction Amounts**: Monetary amounts exhibit extreme positive skew (median: ~$22.00, maximum: $25,691.16). Fraudulent transactions have a higher mean amount but are concentrated in lower-to-mid dollar amounts to evade rule-based thresholds. Log transformation `log1p(Amount)` is essential.
3. **Discriminative PCA Signals**: Features **V14, V17, and V12** demonstrate strong negative correlation with fraud (significant downward shifts in fraud cases), while **V4 and V11** exhibit strong positive correlation.
4. **Diurnal Time Dynamics**: Total transaction volume displays clear 24-hour cyclical patterns (dipping between 03:00 and 06:00 AM). Mapping raw elapsed seconds to cyclical trigonometric features (`hour_sin`, `hour_cos`) preserves continuous periodic boundary conditions.
5. **Data Quality & Scaling**: Zero null values exist across all 30 features. However, severe outliers in `Time` and `Amount` necessitate `RobustScaler` (based on median and IQR) rather than standard Z-score normalization.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to path
sys.path.append(str(Path.cwd().parent))
from src.data.load_data import load_raw_data
from src.config import RAW_DATA_FILE, TARGET_COL

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# Load raw dataset
df = load_raw_data(RAW_DATA_FILE)
print(f"Dataset Shape: {df.shape}")
df.head()

### 1. Class Distribution Analysis

In [ ]:
class_counts = df[TARGET_COL].value_counts()
fraud_pct = (class_counts[1] / len(df)) * 100
print(f"Legitimate (Class 0): {class_counts[0]:,} ({100 - fraud_pct:.3f}%)")
print(f"Fraudulent (Class 1): {class_counts[1]:,} ({fraud_pct:.3f}%)")

fig, ax = plt.subplots(figsize=(6, 4), dpi=150)
sns.barplot(x=class_counts.index, y=class_counts.values, palette=["#2b5c8f", "#d9534f"], ax=ax)
ax.set_yscale('log')
ax.set_xticklabels(['Legitimate (0)', 'Fraud (1)'])
ax.set_ylabel('Transaction Count (Log Scale)')
ax.set_title('Severe Class Imbalance: 0.172% Fraud Rate')
plt.show()

### 2. Transaction Amount Distributions

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), dpi=150)

sns.histplot(df[df[TARGET_COL] == 0]['Amount'], bins=50, ax=ax1, color='#2b5c8f', kde=True)
ax1.set_title('Legitimate Transaction Amounts (Raw)')
ax1.set_xlim(0, 1500)

sns.histplot(np.log1p(df['Amount']), bins=50, ax=ax2, color='#2ca02c', kde=True)
ax2.set_title('All Transactions: log1p(Amount) Distribution')
ax2.set_xlabel('log(1 + Amount)')
plt.tight_layout()
plt.show()

### 3. Correlation with Target (Class)

In [ ]:
corr = df.corr()[TARGET_COL].sort_values()
top_neg = corr.head(6)
top_pos = corr.tail(7).drop(TARGET_COL)
top_corr = pd.concat([top_neg, top_pos])

plt.figure(figsize=(10, 5), dpi=150)
top_corr.plot(kind='barh', color=['#d9534f' if x < 0 else '#2b5c8f' for x in top_corr])
plt.title('Top Positively and Negatively Correlated Features with Fraud (Class)')
plt.xlabel('Pearson Correlation Coefficient')
plt.tight_layout()
plt.show()